# Introduction to Vector Databases (Installing and Running ChromaDB)
**Level:** Scratch → Intermediate**Runs on:** Google Colab (Free CPU)
### What you will learn
1. What a Vector Database is and why we need one
2. Installing ChromaDB (free, open-source, runs locally)
3. Creating a Collection (like a table, but for vectors)
4. Adding documents (ChromaDB auto-generates embeddings)
5. Querying by meaning (semantic search)
6. Using your own Hugging Face embeddings with ChromaDB (intermediate)
7. Updating and deleting records> No API key, no cloud account, no cost. Everything runs in-memory inside this notebook.

## 1. Why a Vector Database?A normal database (SQL) searches by
**exact match**
 — e.g., `WHERE name = 'dog'`.

 A **Vector Database** stores embeddings (numeric meaning-vectors) and searches by **similarity**

  — e.g., "find text that *means* something like 'dog'", even if the word "dog" never appears.

            | Normal Database | Vector Database
            Matches exact text/values | Matches by meaning
            || Uses SQL queries | Uses similarity search
            || Good for structured data | Good for text, images, audio embeddings |
            **ChromaDB** is one of the simplest free, open-source vector databases — great for learning.

### 2. Install ChromaDBRun this once. `-q` keeps the output clean.

In [2]:
!pip install chromadb sentence-transformers

In [3]:
!pip install -q chromadb

### 3. Create a ChromaDB ClientWe use an
**in-memory / ephemeral client**
— perfect for learning in Colab.(Data will reset when the runtime restarts.
For permanent storage, `chromadb.PersistentClient(path="...")` is used instead — shown later.)

In [4]:
import chromadb
client = chromadb.Client()
print("ChromaDB client created successfully!")

ChromaDB client created successfully!


### 4. Create a CollectionA **Collection** in ChromaDB is like a table — it stores your documents, their embeddings, and metadata.

In [5]:
collection = client.create_collection(name="my_first_collection")
print("Collection created:", collection.name)

Collection created: my_first_collection


### 5. Add DocumentsWe simply give ChromaDB plain text. By default, it **automatically converts text into embeddings** for us using a built-in free model — no extra setup needed.

In [6]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers.",
        "Paris is the capital city of France."
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5", "doc6"]
)
print("Documents added! Total documents:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s]


Documents added! Total documents: 6


**Discussion Point:** Every document got an `id` (like a primary key).
Behind the scenes, ChromaDB converted each sentence into an embedding vector and stored it
— just like we did manually in the earlier embeddings notebook.

### 6. Query by Meaning (Semantic Search)Now we ask a question in natural language.
ChromaDB converts our query into an embedding too, then finds the closest matching documents.

In [7]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2)
for doc, distance in zip(results["documents"][0], results["distances"][0]):
  print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 0.9317  |  Deep learning uses neural networks with many layers.
Distance: 0.9516  |  Python is a popular programming language for AI.


**Discussion Point:** Lower `distance` = more similar (closer meaning). Notice the AI/ML related sentences come back first, even though the query never used the exact same words.

In [8]:
# Try another query
results = collection.query(
    query_texts=["What is the capital of France?"],    n_results=2)
for doc, distance in zip(results["documents"][0], results["distances"][0]):
  print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 0.3515  |  Paris is the capital city of France.
Distance: 1.7959  |  Python is a popular programming language for AI.


### 7. Adding Metadata (Intermediate)Real-world documents usually have extra info (category, author, date). ChromaDB lets us store this alongside each document and filter on it.

In [9]:
collection.add(
    documents=[
        "The new smartphone has an amazing camera.",
        "Stock prices rose sharply after the earnings report."
    ],
    metadatas=[
        {"category": "technology"},
         {"category": "finance"}
    ],
    ids=["doc7", "doc8"])
# Query only within the "technology" category
results = collection.query(
    query_texts=["Tell me about new gadgets"],
    n_results=2,    where={"category": "technology"}
)
for doc, distance in zip(results["documents"][0], results["distances"][0]):
  print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 1.1460  |  The new smartphone has an amazing camera.


### 8. Using Your Own Hugging Face Embeddings (Intermediate)By default ChromaDB uses its own built-in embedding model. But we can plug in **any** Hugging Face model, such as `all-MiniLM-L6-v2` from the earlier notebook, for full control.

In [10]:
!pip install -q sentence-transformers

In [11]:
from chromadb.utils import embedding_functions
hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2")
# Create a new collection that uses our chosen Hugging Face model
custom_collection = client.create_collection(
    name="custom_embedding_collection",    embedding_function=hf_embedder
)
custom_collection.add(
    documents=[
        "I love dogs",
        "I like puppies",
        "The stock market crashed today"
    ],
    ids=["a1", "a2", "a3"]
)
results = custom_collection.query(
    query_texts=["puppy lovers"],
    n_results=2)
for doc, distance in zip(results["documents"][0], results["distances"][0]):
  print(f"Distance: {distance:.4f}  |  {doc}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Distance: 0.2513  |  I like puppies
Distance: 0.3943  |  I love dogs


### 9. Update and Delete Records

In [12]:
# Update a document's text
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."]
)
print("Updated doc4")
# Delete a document
collection.delete(ids=["doc2"])
print("Deleted doc2")
print("Remaining documents:", collection.count())

Updated doc4
Deleted doc2
Remaining documents: 7


### 10. Persistent Storage (So Data Survives a Restart)The examples above used an in-memory client, which resets on restart. For real projects, use `PersistentClient` to save data to disk.

In [13]:
persistent_client = chromadb.PersistentClient(path="/content/chroma_db")
persistent_collection = persistent_client.get_or_create_collection(name="persistent_demo")
persistent_collection.add(
    documents=["This data will be saved to disk."],
    ids=["p1"]
)
print("Saved to disk at /content/chroma_db")
print("Count:", persistent_collection.count())

Saved to disk at /content/chroma_db
Count: 1


### 11. Key Takeaways (Recap for Students)
                | Concept | Meaning ||---|---||
                Vector Database | Stores embeddings, searches by meaning/similarity
                || ChromaDB | Free, open-source, runs locally — no cloud needed || Collection | Like a table — holds documents + embeddings + metadata
                || `add()` | Insert documents (embeddings created automatically)
                || `query()` | Search by meaning, returns closest matches
                || `where={}` | Filter results using metadata
                || `PersistentClient` | Saves the database to disk instead of memory |
                ### 12. Practice Exercises
                1. Add 5 of your own sentences to a new collection and query them with a related question.
                2. Add a `"category"` metadata field to every document, then filter a query by two different categories.
                3. Swap the embedding function to `'all-mpnet-base-v2'` and compare the distance scores to `all-MiniLM-L6-v2`.

In [14]:
import chromadb

# Create a persistent ChromaDB client
client = chromadb.PersistentClient(path="./my_chroma_database")

# Create or access a collection
collection = client.get_or_create_collection(
    name="web_development_docs"
)

# Add documents to the collection
collection.upsert(
    ids=["101", "102", "103", "104", "105"],
    documents=[
        "HTML is used to create the structure of web pages.",
        "CSS is used to style and design websites.",
        "JavaScript adds interactive features to web applications.",
        "SQL is used to store and retrieve data from databases.",
        "REST APIs allow different applications to communicate."
    ]
)

# Search query
query = "How can I make a website interactive?"

# Perform semantic search
results = collection.query(
    query_texts=[query],
    n_results=3
)

# Display results
print("Query:", query)
print("\nTop 3 Matching Results:")

for document, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print("Document:", document)
    print("Distance:", round(distance, 4))
    print()

Query: How can I make a website interactive?

Top 3 Matching Results:
Document: JavaScript adds interactive features to web applications.
Distance: 0.7982

Document: HTML is used to create the structure of web pages.
Distance: 1.2173

Document: CSS is used to style and design websites.
Distance: 1.5325



In [15]:
import chromadb

# Create persistent ChromaDB client
client = chromadb.PersistentClient(path="./web_chroma_db")

# Create collection
collection = client.get_or_create_collection(
    name="technology_topics"
)

# Add documents with category metadata
collection.upsert(
    ids=["101", "102", "103", "104", "105", "106"],
    documents=[
        "HTML is used to create the structure of websites.",
        "JavaScript is used to add interaction to web pages.",
        "Football is played by teams around the world.",
        "Tennis is a popular individual sport.",
        "Machine learning helps computers learn from data.",
        "Basketball is played with a ball and a hoop."
    ],
    metadatas=[
        {"category": "web"},
        {"category": "web"},
        {"category": "sports"},
        {"category": "sports"},
        {"category": "AI"},
        {"category": "sports"}
    ]
)

# Search only web development documents
results = collection.query(
    query_texts=["How can I build a website?"],
    n_results=2,
    where={"category": "web"}
)

print("Web Development Results:")

for document in results["documents"][0]:
    print("-", document)


# Search only AI documents
results = collection.query(
    query_texts=["How do computers learn from data?"],
    n_results=1,
    where={"category": "AI"}
)

print("\nAI Results:")

for document in results["documents"][0]:
    print("-", document)

Web Development Results:
- HTML is used to create the structure of websites.
- JavaScript is used to add interaction to web pages.

AI Results:
- Machine learning helps computers learn from data.


In [16]:
import chromadb
from chromadb.utils import embedding_functions

# Create persistent ChromaDB client
client = chromadb.PersistentClient(path="./embedding_models_db")

documents = [
    "HTML is used to create the structure of web pages.",
    "CSS controls the design and appearance of websites.",
    "JavaScript adds interactive behavior to web applications.",
    "SQL is used to manage and retrieve data from databases.",
    "REST APIs allow different applications to exchange information."
]

ids = ["101", "102", "103", "104", "105"]

# MiniLM embedding function
minilm_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# MPNet embedding function
mpnet_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-mpnet-base-v2"
)

# Create separate collections
web_minilm = client.get_or_create_collection(
    name="web_minilm_collection",
    embedding_function=minilm_function
)

web_mpnet = client.get_or_create_collection(
    name="web_mpnet_collection",
    embedding_function=mpnet_function
)

# Store documents in both collections
web_minilm.upsert(
    ids=ids,
    documents=documents
)

web_mpnet.upsert(
    ids=ids,
    documents=documents
)

# Search query
query = "How can I create an interactive website?"

# Search using MiniLM
minilm_results = web_minilm.query(
    query_texts=[query],
    n_results=3
)

# Search using MPNet
mpnet_results = web_mpnet.query(
    query_texts=[query],
    n_results=3
)

# Display MiniLM results
print("===== MiniLM Results =====")

for document, distance in zip(
    minilm_results["documents"][0],
    minilm_results["distances"][0]
):
    print("Document:", document)
    print("Distance:", round(distance, 4))
    print()

# Display MPNet results
print("===== MPNet Results =====")

for document, distance in zip(
    mpnet_results["documents"][0],
    mpnet_results["distances"][0]
):
    print("Document:", document)
    print("Distance:", round(distance, 4))
    print()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

===== MiniLM Results =====
Document: JavaScript adds interactive behavior to web applications.
Distance: 0.4947

Document: HTML is used to create the structure of web pages.
Distance: 0.5867

Document: CSS controls the design and appearance of websites.
Distance: 0.6512

===== MPNet Results =====
Document: JavaScript adds interactive behavior to web applications.
Distance: 0.4625

Document: HTML is used to create the structure of web pages.
Distance: 0.5929

Document: CSS controls the design and appearance of websites.
Distance: 0.6477

